# Extension of the Llama-2-7B vocabulary for the Bashkir language

Goal:

- Train the BPE tokenizer on the `bashqort-raw` corpus (28,600 new tokens).
- Expand the dictionary of the Llama-2-7B model by adding Bashkir tokens.
- Perform a **short additional training** (50 steps) on new embeddings to show that they are integrated. That gives each new token some approximate value, so that later long training begins not from scratch, but from tokens that already “approximately understandable”.
- Save the extended model and tokenizer for use in Experiments A and B.

## Imports & Configs

In [1]:
!pip install -q wandb sentencepiece tokenizers datasets accelerate peft bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00


In [2]:
import os
import torch
import wandb
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, Trainer, TrainingArguments
)
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU name: Tesla T4


In [3]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e278979 (e278979-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN_METU"))

## Load Dataset

In [5]:
from datasets import load_dataset

dataset = load_dataset("metuKKhud/bashqort-raw")

README.md: 0.00B [00:00, ?B/s]

data/bash_news_articles-00000-of-00001.p(…):   0%|          | 0.00/38.5M [00:00<?, ?B/s]

data/bashgazet_articles-00000-of-00001.p(…):   0%|          | 0.00/1.46M [00:00<?, ?B/s]

data/neftcity_articles-00000-of-00001.pa(…):   0%|          | 0.00/451k [00:00<?, ?B/s]

data/public_domain-00000-of-00001.parque(…):   0%|          | 0.00/66.6k [00:00<?, ?B/s]

...

Generating texts_jeshlek split:   0%|          | 0/6259 [00:00<?, ? examples/s]

Generating texts_kiskeufa split:   0%|          | 0/45 [00:00<?, ? examples/s]

Generating texts_kulturarb split:   0%|          | 0/1557 [00:00<?, ? examples/s]

Generating texts_president_rb split:   0%|          | 0/1879 [00:00<?, ? examples/s]

Generating texts_tabin split:   0%|          | 0/539 [00:00<?, ? examples/s]

In [6]:
clean_texts = []
for split_name, split_dataset in dataset.items():
    # is_shuffled == False
    clean_split = split_dataset.filter(lambda x: x['is_shuffled'] is False)
    if len(clean_split) != 0:
        print("Non-shuffled split:", split_name)
    texts_clean = [text.replace('\n', ' ') for text in clean_split['text']]
    clean_texts.extend(texts_clean)
print(f"Overall not-shuffled: {len(clean_texts)}")

Filter:   0%|          | 0/61932 [00:00<?, ? examples/s]

Non-shuffled split: bash_news_articles


Filter:   0%|          | 0/803 [00:00<?, ? examples/s]

Non-shuffled split: bashgazet_articles


Filter:   0%|          | 0/530 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/45 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1557 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1879 [00:00<?, ? examples/s]

Filter:   0%|          | 0/539 [00:00<?, ? examples/s]

Overall not-shuffled: 63270


## Train Tokenizer

In [7]:
def train_bpe_tokenizer(texts, vocab_size=28600):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tokenizer.decoder = decoders.ByteLevel()
    
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
        min_frequency=1, # not skip rare symbols
        show_progress=True
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer

print("Training BPE...")
bpe_tokenizer = train_bpe_tokenizer(clean_texts, vocab_size=28600)
bpe_tokenizer.save("bashkir_bpe_tokenizer.json")

Training BPE...





In [8]:
test_text = "миҙгелдә"
encoded = bpe_tokenizer.encode(test_text)
decoded = bpe_tokenizer.decode(encoded.ids)
print(decoded)

 миҙгелдә


In [9]:
test_paragraph = """Башҡортостанда йәйге каникулдар башланды. Мәктәп уҡыусылары өсөн төрлө ял лагерҙары эшләй. Өфөлә балалар өсөн бушлай мастер-кластар ойошторола. Спорт ярыштары, конкурстар һәм экскурсиялар планлаштырыла.Ата-әсәләр балаларының хәүефһеҙлеген тәьмин итергә тейеш.
"""

unique_chars = set(test_paragraph)
print("Unique symbols:", unique_chars)

problem_chars = []
for ch in unique_chars:
    enc_ch = bpe_tokenizer.encode(ch)
    if '[UNK]' in enc_ch.tokens:
        problem_chars.append(ch)
        print(f"Problem symb: '{ch}' (code U+{ord(ch):04X})")

if not problem_chars:
    print("Ok")
else:
    print(f"Found {len(problem_chars)} problem symbols: {problem_chars}")

Unique symbols: {'ф', 'т', ' ', '-', 'ү', 'л', 'э', 'һ', 'п', 'с', 'ҙ', 'ҡ', '.', 'Ө', 'к', 'ә', 'я', 'й', 'ш', 'у', 'х', '\n', 'е', 'б', 'а', 'ң', 'р', ',', 'г', 'ы', 'н', 'о', 'ө', 'м', 'М', 'ь', 'А', 'д', 'Б', 'и', 'С'}
Problem symb: '
' (code U+000A)
Found 1 problem symbols: ['\n']


In [10]:
test_paragraph_clean = test_paragraph.replace('\n', ' ')
encoded = bpe_tokenizer.encode(test_paragraph_clean)
print(f"[UNK]: {'[UNK]' in encoded.tokens}")
decoded = bpe_tokenizer.decode(encoded.ids)
assert decoded.strip() == test_paragraph_clean.strip()
print("✅")

[UNK]: False
✅


## Model

In [11]:
model_name = "meta-llama/Llama-2-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [12]:
old_tokenizer = AutoTokenizer.from_pretrained(model_name)
old_tokenizer.pad_token = old_tokenizer.eos_token


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_cache=False
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

## Create new tokenizer

In [14]:
hf_bpe_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer,
    pad_token="[PAD]",
    unk_token="[UNK]",
    eos_token=old_tokenizer.eos_token,
    bos_token=old_tokenizer.bos_token
)

In [15]:
new_tokens = list(hf_bpe_tokenizer.get_vocab().keys()) # new bashkort tokens from bpe

num_added = old_tokenizer.add_tokens(new_tokens)
print(f"Numof new added tokens: {num_added}")
print(f"New vocab size: {len(old_tokenizer)}")

Numof new added tokens: 28602
New vocab size: 60379


In [16]:
model.resize_token_embeddings(len(old_tokenizer))
print("Embedding layer expanded to", model.get_input_embeddings().weight.shape[0])

if old_tokenizer.unk_token is None:
    old_tokenizer.unk_token = "[UNK]"

# now old_tokenizer have extended vocab
extended_tokenizer = old_tokenizer

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding layer expanded to 60379


## Train new embeds

This small training allow us to start from feasible loss for continual training and sav gpu hours

In [17]:
sample_texts = clean_texts[:5000]
tokenized_sample = [extended_tokenizer(t, truncation=True, max_length=128, padding="max_length") for t in sample_texts]

In [18]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.input_ids = torch.tensor([e['input_ids'] for e in encodings])
        self.attention_mask = torch.tensor([e['attention_mask'] for e in encodings])
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx], 'attention_mask': self.attention_mask[idx]}

dataloader = DataLoader(TextDataset(tokenized_sample), batch_size=4, shuffle=True)

In [19]:
optimizer = AdamW(model.parameters(), lr=1e-4)
model.train()
device = next(model.parameters()).device

In [20]:
wandb.init(project="bashllama-vocab-extension", name=f"vocab_{len(extended_tokenizer)}_heating")

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260526_111418-11hspffr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vocab_60379_heating
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension/runs/11hspffr


In [21]:
num_steps = 50
log_every = 10
window_size = 10        # for avg

losses = []
step = 0

pbar = tqdm(total=num_steps, desc="Heating embeddings", unit="step")

for batch in dataloader:
    if step >= num_steps:
        break
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = input_ids.clone()
    
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    loss_val = loss.item()
    losses.append(loss_val)
    
    avg_last = sum(losses[-window_size:]) / min(len(losses), window_size)
    pbar.set_postfix({"loss": f"{loss_val:.4f}", "avg10": f"{avg_last:.4f}"})
    pbar.update(1)
    
    if (step + 1) % log_every == 0:
        wandb.log({
            "step": step + 1,
            "loss": loss_val,
            "avg_loss_last10": avg_last
        })
    step += 1

pbar.close()
wandb.log({"final_loss": losses[-1], "loss_curve": wandb.plot.line_series(
    xs=list(range(1, len(losses)+1)), ys=[losses], keys=["loss"], title="Loss over heating steps")})
print(f"Done, final loss: {losses[-1]:.4f}")


Heating embeddings: 100%|██████████| 50/50 [06:49<00:00,  8.19s/step, loss=3.5781, avg10=3.5860]


Done, final loss: 3.5781


Save model

In [22]:
import os
import wandb
import shutil

output_dir = "/kaggle/working/llama2_bashkir_vocab"

model.save_pretrained(output_dir)
extended_tokenizer.save_pretrained(output_dir)
shutil.copy("bashkir_bpe_tokenizer.json", output_dir)

print(f"Model and tokenizer saved into {output_dir}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved into /kaggle/working/llama2_bashkir_vocab


In [23]:
artifact = wandb.Artifact(
    name="llama2_bashkir_vocab",
    type="model",
    description="Llama-2-7B with extended vocabulary for Bashkir (28,600 new tokens, 50-step embedding warmup)",
    metadata={
        "base_model": "meta-llama/Llama-2-7b-hf",
        "original_vocab_size": 32000,
        "new_tokens": len(extended_tokenizer) - 32000,
        "final_vocab_size": len(extended_tokenizer),
        "warmup_steps": 50,
        "warmup_loss": losses[-1] if losses else None
    }
)

artifact.add_dir(output_dir)

wandb.log_artifact(artifact)

wandb.finish()

wandb: Adding directory to artifact (/kaggle/working/llama2_bashkir_vocab)... Done. 31.1s
wandb: uploading artifact llama2_bashkir_vocab; updating run metadata
wandb: uploading artifact llama2_bashkir_vocab
wandb: uploading artifact llama2_bashkir_vocab; uploading summary, console lines 1-1
wandb: uploading artifact llama2_bashkir_vocab
wandb: uploading data
wandb: 
wandb: Run history:
wandb: avg_loss_last10 █▃▅▁▂
wandb:      final_loss ▁
wandb:            loss ▆▁█▃▃
wandb:            step ▁▃▅▆█
wandb: 
wandb: Run summary:
wandb: avg_loss_last10 3.58601
wandb:      final_loss 3.57806
wandb:            loss 3.57806
wandb:            step 50
wandb: 
wandb: 🚀 View run vocab_60379_heating at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension/runs/11hspffr
wandb: ⭐️ View project at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension
wandb: Synced 5 W&B file(s), 1 media file(s), 5 artifact file(s) and 0 other file(s